# Лабораторная работа №12  
# Function Calling и работа с инструментами

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

- Изучить концепцию Function Calling в LLM
- Научиться определять функции для вызова моделью
- Освоить интеграцию с внешними API через инструменты
- Реализовать систему выполнения действий на основе запросов пользователя
- Познакомиться с библиотекой LangChain Tools

---

## 2. Теоретические сведения

### 2.1. Что такое Function Calling

**Function Calling** — механизм, позволяющий языковой модели вызывать внешние функции для получения актуальных данных или выполнения действий.

**Принцип работы:**
1. Пользователь задает вопрос, требующий внешних данных
2. Модель определяет необходимую функцию и её параметры
3. Система выполняет функцию и возвращает результат модели
4. Модель формирует финальный ответ пользователю

### 2.2. Применение Function Calling

| Сценарий | Пример функции |
|----------|----------------|
| Получение погоды | `get_weather(location, date)` |
| Поиск в БД | `search_database(query, filters)` |
| Выполнение кода | `execute_python(code)` |
| Конвертация валют | `convert_currency(amount, from, to)` |
| Календарь | `schedule_meeting(time, participants)` |

### 2.3. Формат описания функций

Функции описываются в формате JSON Schema:

```json
{
  "name": "get_weather",
  "description": "Получить текущую погоду в городе",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string", "description": "Город"},
      "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
    },
    "required": ["location"]
  }
}
```

---

## 3. Задание

### Уровень 1 (Базовый)
1. Установите необходимые библиотеки: `langchain`, `openai` (или используйте локальную модель)
2. Создайте описание 3-х простых функций (калькулятор, конвертер, поиск)
3. Реализуйте парсинг вызовов функций из ответа модели
4. Продемонстрируйте работу на примерах запросов

### Уровень 2 (Продвинутый)
1. Интегрируйте реальное API (например, OpenWeatherMap или курс валют)
2. Реализуйте цепочку вызовов нескольких функций подряд
3. Добавьте обработку ошибок и повторные попытки
4. Создайте мультимодальный интерфейс (текст + результаты функций)

### Уровень 3 (Индивидуальный)
1. Разработайте собственную систему инструментов для конкретной предметной области
2. Реализуйте память между вызовами функций (контекст диалога)
3. Добавьте возможность динамического добавления новых инструментов
4. Оцените точность выбора функций моделью на тестовом наборе

---

## 4. Ход работы

### Шаг 1. Установка библиотек

In [ ]:
!pip install langchain langchain-openai pydantic -q

### Шаг 2. Импорт библиотек и настройка

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.schema import HumanMessage, SystemMessage, AIMessage
import json
import os

### Шаг 3. Определение инструментов (функций)

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка: {str(e)}"

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Конвертирует валюту. Поддерживает: USD, EUR, RUB, GBP."""
    rates = {"USD": 1.0, "EUR": 0.85, "RUB": 90.0, "GBP": 0.73}
    if from_currency not in rates or to_currency not in rates:
        return "Неподдерживаемая валюта"
    usd_amount = amount / rates[from_currency]
    result = usd_amount * rates[to_currency]
    return f"{amount} {from_currency} = {result:.2f} {to_currency}"

@tool
def get_weather(city: str) -> str:
    """Возвращает погоду в городе (симуляция)."""
    import random
    conditions = ["солнечно", "облачно", "дождь", "снег"]
    temp = random.randint(-10, 35)
    condition = random.choice(conditions)
    return f"В городе {city}: {temp}°C, {condition}"

tools = [calculate, convert_currency, get_weather]

### Шаг 4. Тестирование функций

In [ ]:
print(calculate.run({"expression": "2 + 2 * 2"}))
print(convert_currency.run({"amount": 100, "from_currency": "USD", "to_currency": "EUR"}))
print(get_weather.run({"city": "Москва"}))

---

## 5. Контрольные вопросы

1. Объясните принцип работы механизма Function Calling в LLM.
2. Какие преимущества дает использование инструментов?
3. Как описывается схема функции для модели?
4. Какие проблемы безопасности могут возникнуть?
5. Как реализовать обработку ошибок при вызове внешних API?

---

## 6. Выводы

В данной лабораторной работе были изучены основы Function Calling и работы с инструментами в LLM.